## Import Libraries

In [62]:
import pandas as pd
import numpy as np
import time

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve

## Load Data

In [31]:
df = pd.read_excel("diabetic_data.xlsx")
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [32]:
df.shape

(101766, 50)

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  str   
 3   gender                    101766 non-null  str   
 4   age                       101766 non-null  str   
 5   weight                    101766 non-null  str   
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  str   
 11  medical_specialty         101766 non-null  str   
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_medications

In [34]:
df.describe()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [35]:
for col in df.columns:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False))


=== encounter_id ===
encounter_id
2278392      1
149190       1
64410        1
500364       1
16680        1
            ..
443847548    1
443847782    1
443854148    1
443857166    1
443867222    1
Name: count, Length: 101766, dtype: int64

=== patient_nbr ===
patient_nbr
88785891     40
43140906     28
1660293      23
23199021     23
88227540     23
             ..
183087545     1
188574944     1
140199494     1
120975314     1
175429310     1
Name: count, Length: 71518, dtype: int64

=== race ===
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

=== gender ===
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

=== age ===
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)    

## Data Cleaning

#### Input Data Cleaning

In [36]:
# Want to avoid model 'memorising' patient patterns
# Only keep latest encounter_id for duplicate patient_nbr
cleaned_df = (
    df.sort_values('encounter_id', ascending=False)
      .drop_duplicates(subset='patient_nbr', keep='first')
)

cleaned_df = cleaned_df.sort_values('encounter_id').reset_index(drop=True)

In [37]:
# Missing Data
counts = (cleaned_df == '?').sum()
counts[counts > 0]

race                  1878
weight               68671
payer_code           30085
medical_specialty    34525
diag_1                  17
diag_2                 290
diag_3                1146
dtype: int64

In [38]:
### Weight ###

# Drop weight since majority is missing
cleaned_df = cleaned_df.drop(columns=['weight'])

In [39]:
### Payer Code and Medical Specialty ###

# Too many to just add a common value or probalisitic value
# Rather reflect as no value or missing
cleaned_df['payer_code'] = cleaned_df['payer_code'].replace('?', "Missing")
cleaned_df['medical_specialty'] = cleaned_df['medical_specialty'].replace('?', "Missing")

# Drop Payer Code. Has no relevance to readmit
cleaned_df = cleaned_df.drop(columns=['payer_code'])

In [40]:
# Age column is ordinal - encode to median value
encoding = {
    '[90-100)' : 95,  
    '[80-90)'  : 85, 
    '[70-80)'  : 75,
    '[60-70)'  : 65,  
    '[50-60)'  : 55,  
    '[40-50)'  : 45,  
    '[30-40)'  : 35,     
    '[20-30)'  : 25, 
    '[10-20)'  : 15,
    '[0-10)'   : 5 
}

cleaned_df['age'] = cleaned_df['age'].apply(lambda x : encoding[x])

In [41]:
# Drop singe-value columns as they provide no variance and hence do not aid the classification problem
cleaned_df =  cleaned_df.drop(columns=['examide', 'citoglipton', 'glimepiride-pioglitazone'])

In [42]:
# Covert binary columns to 0/1
binary_map = {
    "gender": {"Female": 0, "Male": 1},
    "change": {"No": 0, "Ch": 1},
    "diabetesMed": {"No": 0, "Yes": 1}
}

for col, mapping in binary_map.items():
    cleaned_df[col] = cleaned_df[col].map(mapping)

In [43]:
# medication columns use ordinal encoding
medication_map = {"No": 0, "Steady": 1, "Up": 2, "Down": 3}

medication_cols = [
    "metformin","repaglinide","nateglinide","chlorpropamide",
    "glimepiride","acetohexamide","glipizide","glyburide","tolbutamide","pioglitazone",
    "rosiglitazone","acarbose","miglitol","troglitazone",
    "tolazamide","insulin","glyburide-metformin",
    "glipizide-metformin","metformin-rosiglitazone",
    "metformin-pioglitazone"
]

for col in medication_cols:
    cleaned_df[col] = cleaned_df[col].map(medication_map)

#### Target encoding

In [44]:
# We only interested in if they readmit within 30 days
readmit_map = {">30": 0, "NO": 0, "<30": 1}
cleaned_df["readmitted"] = cleaned_df["readmitted"].map(readmit_map)

In [45]:
cleaned_df['readmitted'].value_counts(normalize=True)

readmitted
0    0.95492
1    0.04508
Name: proportion, dtype: float64

In [46]:
for col in cleaned_df.columns:
    print(f"\n=== {col} ===")
    print(cleaned_df[col].value_counts(dropna=False))


=== encounter_id ===
encounter_id
12522        1
15738        1
16680        1
28236        1
35754        1
            ..
443847548    1
443847782    1
443854148    1
443857166    1
443867222    1
Name: count, Length: 71518, dtype: int64

=== patient_nbr ===
patient_nbr
48330783     1
63555939     1
42519267     1
89869032     1
82637451     1
            ..
100162476    1
74694222     1
41088789     1
31693671     1
175429310    1
Name: count, Length: 71518, dtype: int64

=== race ===
race
Caucasian          53538
AfricanAmerican    12915
?                   1878
Hispanic            1508
Other               1167
Asian                512
Name: count, dtype: int64

=== gender ===
gender
0.0    38023
1.0    33492
NaN        3
Name: count, dtype: int64

=== age ===
age
75    18162
65    15908
55    12349
85    11864
45     6756
35     2650
95     2040
25     1111
15      525
5       153
Name: count, dtype: int64

=== admission_type_id ===
admission_type_id
1    37191
3    13823
2    13

## Split data

In [47]:
# TimeSeriesSplit only gives you two splits
# We need 3: train, val and test
def time_series_split_3(df, n_splits=5, val_size=0.15, test_size=0.15):
    n = len(df)
    val_len = int(n * val_size)
    test_len = int(n * test_size)

    step = (n - val_len - test_len) // n_splits

    for i in range(1, n_splits + 1):
        train_end = step * i
        val_end = train_end + val_len
        test_end = val_end + test_len

        train = df.iloc[:train_end]
        val = df.iloc[train_end:val_end]
        test = df.iloc[val_end:test_end]

        yield train, val, test

## Make Model

In [48]:
# Create input and target
X_cols = cleaned_df.columns.drop(['encounter_id', 'patient_nbr', 'readmitted']).tolist()
y_col = 'readmitted'

In [49]:
# Train/Val/Test specific cleaning
# These columns need to be cleaned in each set to prevent data leakage

### Race ###
# Impute Race using probability of other types

### Diagnoses ###
# Fill diag_1, diag_2 and diag_3 with the most common value
# They have entries such as 'V45' hence we can't use median or mean
# Implement frequency encoding since each one of these cols have over 700 unique values

### Medical Speciality ###
# medical_speciality has a lot of missing, keep only top 10, group rest

class DataCleaner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.race_probabilities_ = None
        self.diag_common_ = {}
        self.diag_frequencies_ = {}
        self.top_med_specialties_ = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # 1. Learn Race Probabilities
        race_col = X['race']
        valid_race = race_col[race_col != '?']
        self.race_probabilities_ = valid_race.value_counts(normalize=True)

        # 2. Learn Diagnosis Modes & Frequencies
        for col in ["diag_1", "diag_2", "diag_3"]:
            valid_diags = X.loc[X[col] != '?', col]
            self.diag_common_[col] = valid_diags.mode().iloc[0]                                 # Get most common diagnoses
            self.diag_frequencies_[col] = valid_diags.value_counts(normalize=True)              # Get frequencies of diagnoses

        # 3. Learn Top 10 Specialties
        self.top_med_specialties_ = X['medical_specialty'].value_counts().nlargest(10).index
        
        return self

    def transform(self, X):
        X = X.copy()

        # --- Apply Race Imputation ---
        race_empty = X['race'] == '?'
        if race_empty.any():
            X.loc[race_empty, 'race'] = np.random.choice(
                self.race_probabilities_.index,
                size=race_empty.sum(),
                p=self.race_probabilities_.values
            )

        # --- Apply Diagnosis Cleaning ---
        for col in ["diag_1", "diag_2", "diag_3"]:
            # Replace '?' with most common value
            X[col] = X[col].replace('?', self.diag_common_[col])
            # Map frequencies learned in fit (unseen values get 0)
            X[col] = X[col].map(self.diag_frequencies_[col]).fillna(0)

        # --- Apply Medical Specialty Grouping ---
        X['medical_specialty'] = X['medical_specialty'].where(
            X['medical_specialty'].isin(self.top_med_specialties_), 
            'Other'
        )

        return X

In [ ]:
def make_model(dummyclassifier=False, learning_rate=1e-4):
    # Normalize numeric columns
    numeric_cols = [
        "age","time_in_hospital","num_lab_procedures","num_procedures",
        "num_medications","number_outpatient","number_emergency",
        "number_inpatient","diag_1","diag_2","diag_3","number_diagnoses"
    ]

    # One-hot encode categorical columns with few unique values and no order
    categorical_cols = [
        "race","admission_type_id","discharge_disposition_id","admission_source_id", 'medical_specialty', "max_glu_serum","A1Cresult"
    ]

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ])
    
    if dummyclassifier:
        model = DummyClassifier(strategy="prior",random_state=42)
    else:
        model = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=learning_rate, class_weight="balanced", random_state=42)

    return Pipeline([
        ("cleaner", DataCleaner()),
        ("prep", preprocessor),
        ("model", model)
    ])

In [51]:
def run_model(dummyclassifier=False, learning_rate=1e-4):
    start_time = time.perf_counter()
    results = []

    for fold, (train_df, val_df, test_df) in enumerate(
            time_series_split_3(cleaned_df, n_splits=5, val_size=0.15, test_size=0.15)):

        X_train, y_train = train_df[X_cols], train_df[y_col]
        X_val, y_val = val_df[X_cols], val_df[y_col]
        X_test, y_test = test_df[X_cols], test_df[y_col]

        model = make_model(dummyclassifier=dummyclassifier, learning_rate=learning_rate)
        model.fit(X_train, y_train)

        # Get probabilities not class labels
        y_train_proba = model.predict_proba(X_train)[:, 1]
        y_val_proba = model.predict_proba(X_val)[:, 1]
        y_test_proba = model.predict_proba(X_test)[:, 1]

        # Compute ROC AUC
        roc_train = roc_auc_score(y_train, y_train_proba)
        roc_val = roc_auc_score(y_val, y_val_proba)
        roc_test = roc_auc_score(y_test, y_test_proba)

        # Compute PR AUC
        pr_train = average_precision_score(y_train, y_train_proba)
        pr_val = average_precision_score(y_val, y_val_proba)
        pr_test = average_precision_score(y_test, y_test_proba)

        # Get Recall at Precision >= 0.8 for Val
        precision, recall, thresholds = precision_recall_curve(y_val, y_val_proba)
        valid = precision[:-1] >= 0.8

        if np.any(valid):
            best_recall = recall[:-1][valid].max()
            print("Recall at Precision ≥ 0.8:", best_recall)
        else:
            print("Precision never reaches 0.8")

        results.append((fold, roc_train, pr_train, roc_val, pr_val, roc_test, pr_test))

    end_time = time.perf_counter()
    runtime = end_time - start_time

    return results, runtime

## Baseline Model

In [56]:
print("Baseline Classifier \n")

results, runtime = run_model(dummyclassifier=True)
results_df = pd.DataFrame(results, columns=["fold", "roc_train", "pr_train", "roc_val", "pr_val", "roc_test", "pr_test"])

print(f'\n Runtime: {runtime} s \n')
print(results_df)
print("\n Mean test ROC_AUC:", results_df["roc_test"].mean())
print("Mean test PR_AUC:", results_df["pr_test"].mean())

Baseline Classifier 

Precision never reaches 0.8
Precision never reaches 0.8
Precision never reaches 0.8
Precision never reaches 0.8
Precision never reaches 0.8

 Runtime: 1.5239611000288278 s 

   fold  roc_train  pr_train  roc_val    pr_val  roc_test   pr_test
0     0        0.5  0.047343      0.5  0.050993       0.5  0.059476
1     1        0.5  0.048592      0.5  0.059290       0.5  0.038687
2     2        0.5  0.051838      0.5  0.042137       0.5  0.031975
3     3        0.5  0.049391      0.5  0.032069       0.5  0.024611
4     4        0.5  0.045925      0.5  0.025636       0.5  0.060595

 Mean test ROC_AUC: 0.5
Mean test PR_AUC: 0.04306889158198938


## SGD Sensitivity: 3 Fixed Learning Rates 

In [57]:
# eta0 = 1e-4
print("SGD Classifier with eta0=1e-4 \n")
results, runtime = run_model(learning_rate=1e-4)
results_df = pd.DataFrame(results, columns=["fold", "roc_train", "pr_train", "roc_val", "pr_val", "roc_test", "pr_test"])

print(f'\nRuntime: {runtime} s \n')
print(results_df)
print("\nMean train PR_AUC:", results_df["pr_train"].mean())
print("Mean val PR_AUC:", results_df["pr_val"].mean())
print("Mean test PR_AUC:", results_df["pr_test"].mean())

SGD Classifier with eta0=1e-4 

Precision never reaches 0.8
Precision never reaches 0.8
Recall at Precision ≥ 0.8: 0.0022123893805309734
Recall at Precision ≥ 0.8: 0.0029069767441860465
Precision never reaches 0.8

Runtime: 2.257864800048992 s 

   fold  roc_train  pr_train   roc_val    pr_val  roc_test   pr_test
0     0   0.764991  0.143585  0.708190  0.124012  0.701136  0.139519
1     1   0.760846  0.145819  0.728742  0.147875  0.691603  0.089577
2     2   0.746475  0.144951  0.676955  0.095891  0.671024  0.064014
3     3   0.745310  0.140834  0.708691  0.081498  0.667618  0.047883
4     4   0.725790  0.121494  0.689866  0.059387  0.639881  0.105895

Mean train PR_AUC: 0.13933668128437798
Mean val PR_AUC: 0.10173265921275917
Mean test PR_AUC: 0.08937766825823591


In [58]:
# eta0 = 1e-3
print("SGD Classifier with eta0=1e-3 \n")
results, runtime = run_model(learning_rate=1e-3)
results_df = pd.DataFrame(results, columns=["fold", "roc_train", "pr_train", "roc_val", "pr_val", "roc_test", "pr_test"])

print(f'\nRuntime: {runtime} s \n')
print(results_df)
print("\nMean train PR_AUC:", results_df["pr_train"].mean())
print("Mean val PR_AUC:", results_df["pr_val"].mean())
print("Mean test PR_AUC:", results_df["pr_test"].mean())

SGD Classifier with eta0=1e-3 

Precision never reaches 0.8
Precision never reaches 0.8
Recall at Precision ≥ 0.8: 0.0022123893805309734
Recall at Precision ≥ 0.8: 0.0029069767441860465
Precision never reaches 0.8

Runtime: 2.164649099984672 s 

   fold  roc_train  pr_train   roc_val    pr_val  roc_test   pr_test
0     0   0.790596  0.159988  0.724317  0.135854  0.707809  0.140542
1     1   0.781191  0.163745  0.736173  0.149921  0.724066  0.099633
2     2   0.764486  0.160304  0.704771  0.106344  0.690162  0.071012
3     3   0.758144  0.145227  0.718139  0.080694  0.677428  0.046571
4     4   0.748722  0.134112  0.704674  0.062307  0.643868  0.109514

Mean train PR_AUC: 0.15267511113119586
Mean val PR_AUC: 0.10702398432186344
Mean test PR_AUC: 0.09345447303134177


In [59]:
# eta0 = 1e-2
print("SGD Classifier with eta0=1e-2 \n")
results, runtime = run_model(learning_rate=1e-2)
results_df = pd.DataFrame(results, columns=["fold", "roc_train", "pr_train", "roc_val", "pr_val", "roc_test", "pr_test"])

print(f'\nRuntime: {runtime} s \n')
print(results_df)
print("\nMean train PR_AUC:", results_df["pr_train"].mean())
print("Mean val PR_AUC:", results_df["pr_val"].mean())
print("Mean test PR_AUC:", results_df["pr_test"].mean())

SGD Classifier with eta0=1e-2 

Precision never reaches 0.8
Precision never reaches 0.8
Precision never reaches 0.8
Recall at Precision ≥ 0.8: 0.0029069767441860465
Precision never reaches 0.8

Runtime: 1.9061126999440603 s 

   fold  roc_train  pr_train   roc_val    pr_val  roc_test   pr_test
0     0   0.749019  0.142269  0.668049  0.108565  0.637271  0.111831
1     1   0.760646  0.136914  0.722470  0.127962  0.696496  0.078383
2     2   0.734099  0.133092  0.693110  0.089383  0.646173  0.052480
3     3   0.727216  0.127771  0.684716  0.079632  0.646055  0.044617
4     4   0.682779  0.110951  0.642108  0.053242  0.608758  0.098745

Mean train PR_AUC: 0.13019947114504765
Mean val PR_AUC: 0.09175680798008694
Mean test PR_AUC: 0.0772110162257846


## Grid Search over SGD Hyperparameters

In [ ]:
# Define parameter Grid

param_grid = {
    "model__eta0": [1e-4,3e-4,1e-3,3e-3,1e-2],
    "model__alpha": [1e-6,1e-5,1e-4,1e-3],
    "model__penalty":["l2", "l1"],
    "model__class_weight": [None, "balanced"]
}